In [ ]:
# MOUNT GOOGLE DRIVE

import pandas as pd
import requests
import zipfile
import io
import os
import warnings
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')


Mounting Google Drive...
Mounted at /content/drive


In [ ]:
!pip install netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.7 MB/s eta 0:00:00


In [ ]:
!pip install copernicusmarine -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.1 MB/s eta 0:00:00


In [ ]:
import copernicusmarine
help(copernicusmarine)

In [ ]:
import copernicusmarine

dataset_id = "cmems_mod_glo_phy_anfc_0.083deg_PT1H-m"

print(f" Inspecting Variables ")

try:
    catalogue = copernicusmarine.describe(dataset_id=dataset_id)

    for product in catalogue.products:
        for dataset in product.datasets:
            if dataset.dataset_id == dataset_id:
                print(f"Dataset Found: {dataset.dataset_name}\n")
                print(f"{'Short Name':<15} | {'Standard Name'}")
                # Correctly accessing the variables attribute from the catalogue model
                for version in dataset.versions:
                    for part in version.parts:
                        for service in part.services:
                            if service.variables:
                                for var in service.variables:
                                    print(f"{var.short_name:<15} | {var.standard_name}")

except Exception as e:
    print(f" Failed : {e}")

print("\n Completed ")

 Inspecting Variables 


Fetching catalogue 1: 100%|██████████| 2/2 [00:02<00:00,  1.08s/it]

Dataset Found: hourly mean fields from Global Ocean Physics Analysis and Forecast updated Daily

Short Name      | Standard Name
so              | sea_water_salinity
thetao          | sea_water_potential_temperature
uo              | eastward_sea_water_velocity
vo              | northward_sea_water_velocity
zos             | sea_surface_height_above_geoid
so              | sea_water_salinity
thetao          | sea_water_potential_temperature
uo              | eastward_sea_water_velocity
vo              | northward_sea_water_velocity
zos             | sea_surface_height_above_geoid
so              | sea_water_salinity
thetao          | sea_water_potential_temperature
uo              | eastward_sea_water_velocity
vo              | northward_sea_water_velocity
zos             | sea_surface_height_above_geoid
so              | sea_water_salinity
thetao          | sea_water_potential_temperature
uo              | eastward_sea_water_velocity
vo              | northward_sea_water_velocity
zos 

In [ ]:
import copernicusmarine
import xarray as xr
import os
import gc
from google.colab import drive
import calendar

client = copernicusmarine.login(username=USR, password=PWD)

lon_bounds, lat_bounds = [-112, -60], [9, 52]
months = [f"{i:02d}" for i in range(1, 13)]

save_path = "/content/drive/MyDrive/LCO2 Transport Project Final"
YEAR = 2025

def last_day_of_month(year, month):
    #Correct last day (28-31) for a given month or year.
    return calendar.monthrange(year, month)[1]

print("Starting Process..")

for month in months:
    print(f"\n Cuurently Processing Month: {month}")

    try:
        last_day = last_day_of_month(YEAR, int(month))

        # PHYSICS
        ds_phy = copernicusmarine.open_dataset(
            dataset_id="cmems_mod_glo_phy_anfc_0.083deg_PT1H-m",
            variables=["uo", "vo", "thetao"],
            start_datetime=f"{YEAR}-{month}-01T00:00:00",
            end_datetime=f"{YEAR}-{month}-{last_day:02d}T23:59:59",
            minimum_longitude=lon_bounds[0], maximum_longitude=lon_bounds[1],
            minimum_latitude=lat_bounds[0], maximum_latitude=lat_bounds[1],
        )

        # WAVES
        ds_wav = copernicusmarine.open_dataset(
            dataset_id="cmems_mod_glo_wav_anfc_0.083deg_PT3H-i",
            variables=["VHM0", "VMDR", "VHM0_WW"],
            start_datetime=f"{YEAR}-{month}-01T00:00:00",
            end_datetime=f"{YEAR}-{month}-{last_day:02d}T23:59:59",
            minimum_longitude=lon_bounds[0], maximum_longitude=lon_bounds[1],
            minimum_latitude=lat_bounds[0], maximum_latitude=lat_bounds[1],
        )

        # RENAME & INTERPOLATE
        ds_wav = ds_wav.rename({"VHM0": "swh", "VMDR": "mwd", "VHM0_WW": "wind_wave_height"})
        ds_wav_hourly = ds_wav.reindex_like(ds_phy, method="ffill")

        # MERGE & SAVE
        ds_final = xr.merge([ds_phy, ds_wav_hourly])
        output_file = os.path.join(save_path, f"gulf_merged_{YEAR}_{month}.nc")

        comp = dict(zlib=True, complevel=5)
        encoding = {var: comp for var in ds_final.data_vars}

        # write to netcdf
        ds_final.to_netcdf(output_file, encoding=encoding)
        print("ds_final.to_netcdf(output_file, encoding=encoding) - command executed")

        # Flush to Drive
        drive.flush_and_unmount()
        drive.mount('/content/drive') # Remount for the next iteration

        print(f" SUCCESS! Saved to {output_file}")

    except Exception as e:
        print(f"FAILED Month {month}: {e}")
        continue

    # CLEANUP

    try:
        ds_phy.close()
        ds_wav.close()
        ds_final.close()
    except:
        pass

    del ds_phy, ds_wav, ds_wav_hourly, ds_final
    gc.collect() # Clear RAM
    print(f" RAM Cleared for Month {month}")

print("\n Done! ")

INFO - 2026-09-02T17:22:30Z - Credentials file stored in /root/.copernicusmarine/.copernicusmarine-credentials.
INFO:copernicusmarine:Credentials file stored in /root/.copernicusmarine/.copernicusmarine-credentials.


Starting Process..

 Cuurently Processing Month: 01


INFO - 2026-09-02T17:22:34Z - Selected dataset version: "202406"
INFO:copernicusmarine:Selected dataset version: "202406"
INFO - 2026-09-02T17:22:34Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
INFO - 2026-09-02T17:22:41Z - Selected dataset version: "202411"
INFO:copernicusmarine:Selected dataset version: "202411"
INFO - 2026-09-02T17:22:41Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


ds_final.to_netcdf(output_file, encoding=encoding) - command executed
Mounted at /content/drive
 SUCCESS! Saved to /content/drive/MyDrive/LCO2 Transport Project Final/gulf_merged_2025_01.nc
 RAM Cleared for Month 01

 Cuurently Processing Month: 02


INFO - 2026-09-02T17:38:38Z - Selected dataset version: "202406"
INFO:copernicusmarine:Selected dataset version: "202406"
INFO - 2026-09-02T17:38:38Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
INFO - 2026-09-02T17:38:45Z - Selected dataset version: "202411"
INFO:copernicusmarine:Selected dataset version: "202411"
INFO - 2026-09-02T17:38:45Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


...

ds_final.to_netcdf(output_file, encoding=encoding) - command executed
Mounted at /content/drive
 SUCCESS! Saved to /content/drive/MyDrive/LCO2 Transport Project Final/gulf_merged_2025_10.nc
 RAM Cleared for Month 10

 Cuurently Processing Month: 11


INFO - 2026-09-02T20:00:21Z - Selected dataset version: "202406"
INFO:copernicusmarine:Selected dataset version: "202406"
INFO - 2026-09-02T20:00:21Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
INFO - 2026-09-02T20:00:27Z - Selected dataset version: "202411"
INFO:copernicusmarine:Selected dataset version: "202411"
INFO - 2026-09-02T20:00:27Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


ds_final.to_netcdf(output_file, encoding=encoding) - command executed
Mounted at /content/drive
 SUCCESS! Saved to /content/drive/MyDrive/LCO2 Transport Project Final/gulf_merged_2025_11.nc
 RAM Cleared for Month 11

 Cuurently Processing Month: 12


INFO - 2026-09-02T20:14:42Z - Selected dataset version: "202406"
INFO:copernicusmarine:Selected dataset version: "202406"
INFO - 2026-09-02T20:14:42Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
INFO - 2026-09-02T20:14:48Z - Selected dataset version: "202411"
INFO:copernicusmarine:Selected dataset version: "202411"
INFO - 2026-09-02T20:14:48Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


ds_final.to_netcdf(output_file, encoding=encoding) - command executed
Mounted at /content/drive
 SUCCESS! Saved to /content/drive/MyDrive/LCO2 Transport Project Final/gulf_merged_2025_12.nc
 RAM Cleared for Month 12

 Done! 


In [ ]:
import os

file_path = "/content/drive/MyDrive/LCO2 Transport Project Final/cleaned_noaa_fleet_2025.csv"

df = pd.read_csv(file_path, compression='gzip')
df.columns

Index(['BaseDateTime', 'LON', 'LAT', 'SOG', 'COG', 'Heading', 'IMO', 'Status',
       'Length', 'Width', 'Draft', 'Cargo', 'drift_angle'],
      dtype='object')

In [ ]:
df['BaseDateTime']

,BaseDateTime
0,2025-01-01 00:00:02
1,2025-01-01 00:00:00
2,2025-01-01 00:00:04
3,2025-01-01 00:00:00
4,2025-01-01 00:00:00
...,...
2606723,2025-12-31 19:28:57
2606724,2025-12-31 19:48:59
2606725,2025-12-31 21:54:02
2606726,2025-12-31 22:57:13


In [ ]:
import pandas as pd
import xarray as xr
import os

csv_path = "/content/drive/MyDrive/LCO2 Transport Project Final/cleaned_noaa_fleet_2025.csv"
nc_pattern = "/content/drive/MyDrive/LCO2 Transport Project Final/gulf_merged_2025_*.nc"

#  LOADING AIS DATA
print("Attempting to decompress and read AIS data.")
try:
    # reading a gzipped file
    df_ais = pd.read_csv(csv_path, compression='gzip', low_memory=False)
    print(f"Success! Loaded {len(df_ais)} rows.")
except Exception as e:
    print(f"Gzip failed")
    df_ais = pd.read_csv(csv_path, on_bad_lines='skip', engine='python')

# MERGE
df_ais['BaseDateTime'] = pd.to_datetime(df_ais['BaseDateTime'])

print("Opening NetCDF files")
ds = xr.open_mfdataset(nc_pattern, combine='by_coords')

# Prepare coordinate mapping
query_time = xr.DataArray(df_ais['BaseDateTime'], dims="z")
query_lat = xr.DataArray(df_ais['LAT'], dims="z")
query_lon = xr.DataArray(df_ais['LON'], dims="z")

print("Using nearest neighbours")
# Select nearest grid points
ds_points = ds.sel(
    time=query_time,
    latitude=query_lat,
    longitude=query_lon,
    method='nearest'
).load()

# Join back
df_env = ds_points.to_dataframe().reset_index()
df_final = pd.concat([df_ais.reset_index(drop=True),
                      df_env.drop(columns=['time', 'latitude', 'longitude'])], axis=1)

print(df_final.head())

Attempting to decompress and read AIS data.
Success! Loaded 365318 rows.
Opening NetCDF files
Using nearest neighbours
         BaseDateTime       LON       LAT   SOG    COG  Heading         IMO  \
0 2025-01-01 00:00:04 -90.29858  29.95597  14.3   40.7     41.0  IMO7816551   
1 2025-01-01 00:00:03 -89.41682  29.35364  13.1  125.7    128.0  IMO9885908   
2 2025-01-01 00:00:02 -95.20078  29.74000   4.6   52.9     54.0  IMO9724611   
3 2025-01-01 00:00:07 -91.00400  30.16538  10.6  338.0    335.0  IMO9675078   
4 2025-01-01 00:00:04 -95.80819  27.19427  11.6  114.9    116.0  IMO9681857   

   Status  Length  Width  ...  Cargo  drift_angle  z     depth        uo  \
0     0.0   192.0   32.0  ...   80.0          0.3  0  0.494025       NaN   
1     0.0   229.0   35.0  ...   80.0          2.3  1  0.494025       NaN   
2     0.0   182.0   32.0  ...   80.0          1.1  2  0.494025       NaN   
3     0.0   147.0   23.0  ...   81.0         -3.0  3  0.494025       NaN   
4     0.0   183.0   32.0  

In [ ]:
output_path = "/content/drive/MyDrive/LCO2 Transport Project Final/uncleaned_ais_copernicus_merged_dataset_2025.csv"
df_final.to_csv(output_path, index=False)
print("Dataset saved successfully.")

Dataset saved successfully.
